
# 15 — Advanced LangGraph: Human Oversight and Time Travel

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Adapted from these course notebooks:**

- `tool_usage_and_persistence_in_langgraph.ipynb`
- `human_in_the_loop.ipynb`
- `manually_updating_the_state.ipynb`
- `customizing_state.ipynb`
- `time_travel.ipynb`

**Related foundational lesson:**
- `simple_langgraph.ipynb`

**Additional reference:**
[LangGraph documentation](https://docs.langchain.com/oss/python/langgraph/overview)

## Goal

Build an advanced LangGraph workflow that supports:

1. Tool execution inside a graph.
2. Pausing execution for human review.
3. Approving or rejecting a proposed action.
4. Modifying tool arguments before execution.
5. Inspecting and updating saved workflow state.
6. Keeping independent execution threads.
7. Viewing checkpoint history.
8. Replaying a workflow from an earlier checkpoint.
9. Understanding in-memory and SQLite persistence.

## Relationship to Notebook 14

Notebook 14 introduced graph nodes, shared state,
conditional transitions, validation loops and
checkpointed conversation history.

This notebook focuses on controlling an execution
after it has started.

The examples use deterministic Python logic rather
than a language model. Tool requests are constructed
locally, so the notebook demonstrates LangGraph
execution mechanics without claiming an LLM
selected the tools.

No external AI API calls are required.



## 1. Why Human Oversight Matters

An AI agent may propose actions such as:

- Looking up sensitive information.
- Editing a document.
- Sending a message.
- Processing an order.
- Changing a database record.

Not every action should run automatically.

A human-in-the-loop workflow allows a person to
review a proposed action before it executes.

LangGraph supports this using interruptions
and saved workflow state.

An important distinction:

**Interrupt:** Pause execution and request input.

**Resume:** Continue after receiving a decision.

**State update:** Change information stored in
the workflow.

**Checkpoint:** A saved snapshot of workflow state.

**Replay:** Continue execution from a previously
saved checkpoint.

A checkpoint preserves computational state.
It does not undo external actions that have already
occurred.


In [1]:

%pip install -q "langgraph>=1,<2" "langchain-core>=1,<2" "langgraph-checkpoint-sqlite>=3,<4"


In [4]:

from typing import Annotated, TypedDict
from operator import add

from langchain_core.messages import (
    AIMessage,
    ToolMessage
)
from langgraph.graph.message import add_messages
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver

POLICIES = {
    "returns": "Unused items can be returned within 30 days.",
    "delivery": "Standard delivery takes 3–5 business days.",
    "damage": "Customers can request help with damaged items.",
    "general": "Contact customer support for further assistance."
}

print("Environment ready.")
print("Policy categories:", list(POLICIES))
print("Model API requests: 0")


Environment ready.
Policy categories: ['returns', 'delivery', 'damage', 'general']
Model API requests: 0



## 2. Define the Tool and Custom State

We will build a small support-policy assistant.

The workflow proposes a category based on the
customer's message.

For example:

"My item arrived damaged."

The workflow proposes calling:

`lookup_policy(category="damage")`

Before executing that tool, the graph pauses.

A human may:

- Approve the proposed category.
- Change it to another category.
- Reject the action entirely.

### Custom State

The graph stores:

- `request`: Original customer message.
- `proposed_category`: Selected policy category.
- `messages`: Tool requests and responses.
- `approved`: Human approval status.
- `outcome`: Final workflow result.
- `audit`: An ordered log of relevant actions.

The message reducer allows an existing message to
be replaced when another message uses the same ID.

This becomes useful when correcting a tool request.


In [5]:

@tool
def lookup_policy(category: str) -> str:
    """Retrieve the local customer-support policy for a category."""
    return POLICIES.get(
        category,
        "No matching policy was found."
    )


class ReviewState(TypedDict, total=False):
    request: str
    proposed_category: str
    messages: Annotated[list, add_messages]
    approved: bool
    outcome: str
    audit: Annotated[list[str], add]


print("Tool:", lookup_policy.name)
print("Tool arguments:", lookup_policy.args)
print("Custom state defined.")


Tool: lookup_policy
Tool arguments: {'category': {'title': 'Category', 'type': 'string'}}
Custom state defined.



## 3. Propose, Review and Execute

We will create four graph nodes.

**Propose**

Uses deterministic rules to select a policy category
and constructs an AI-style tool request.

**Approve**

Pauses the graph with `interrupt()` and requests
a human decision.

**Tools**

Uses `ToolNode` to execute the approved tool call.

**Finalize**

Reads the actual tool result and produces the
final workflow outcome.

The graph uses conditional routing:

- Approved → Tools → Finalize → END
- Rejected → END

A request rejected during approval never reaches
the tool-execution node.

This is an important safety property.


In [6]:

def propose_action(state: ReviewState):
    text = state["request"].lower()

    if "return" in text or "refund" in text:
        category = "returns"
    elif "damage" in text or "broken" in text:
        category = "damage"
    elif "deliver" in text or "late" in text:
        category = "delivery"
    else:
        category = "general"

    proposal = AIMessage(
        content=f"Proposed policy lookup: {category}",
        tool_calls=[
            {
                "name": "lookup_policy",
                "args": {"category": category},
                "id": "policy_lookup_1",
                "type": "tool_call"
            }
        ]
    )

    return {
        "proposed_category": category,
        "messages": [proposal],
        "audit": [f"Proposed policy category: {category}"]
    }


def human_approval(state: ReviewState):
    # The interrupt happens before any tool executes.
    # This node will restart here when resumed.
    decision = interrupt({
        "question": "Approve the proposed policy lookup?",
        "request": state["request"],
        "proposed_category": state["proposed_category"],
        "allowed_categories": list(POLICIES)
    })

    if not isinstance(decision, dict):
        raise ValueError("Human decision must be a dictionary.")

    approved = decision.get("approved")

    if type(approved) is not bool:
        raise ValueError("'approved' must be True or False.")

    if not approved:
        return {
            "approved": False,
            "outcome": "Action rejected by human reviewer.",
            "audit": ["Human rejected the proposed action"]
        }

    category = decision.get(
        "category",
        state["proposed_category"]
    )

    if category not in POLICIES:
        raise ValueError("Unsupported policy category.")

    # Replace the pending AI message with a corrected
    # tool call, preserving its message ID.
    previous_message = state["messages"][-1]

    approved_message = AIMessage(
        id=previous_message.id,
        content=f"Approved policy lookup: {category}",
        tool_calls=[
            {
                "name": "lookup_policy",
                "args": {"category": category},
                "id": "policy_lookup_1",
                "type": "tool_call"
            }
        ]
    )

    return {
        "approved": True,
        "proposed_category": category,
        "messages": [approved_message],
        "audit": [f"Human approved category: {category}"]
    }


def choose_approval_path(state: ReviewState):
    if state.get("approved") is True:
        return "tools"

    return "finish"


def finalize_result(state: ReviewState):
    tool_results = [
        message
        for message in state["messages"]
        if isinstance(message, ToolMessage)
    ]

    if not tool_results:
        raise RuntimeError("No tool result available.")

    policy = str(tool_results[-1].content)

    return {
        "outcome": policy,
        "audit": ["Tool result received and finalized"]
    }


print("Workflow functions ready.")


Workflow functions ready.


In [7]:

builder = StateGraph(ReviewState)

builder.add_node("propose", propose_action)
builder.add_node("approve", human_approval)
builder.add_node(
    "tools",
    ToolNode([lookup_policy])
)
builder.add_node("finalize", finalize_result)

builder.add_edge(START, "propose")
builder.add_edge("propose", "approve")

builder.add_conditional_edges(
    "approve",
    choose_approval_path,
    {
        "tools": "tools",
        "finish": END
    }
)

builder.add_edge("tools", "finalize")
builder.add_edge("finalize", END)

memory = InMemorySaver()

graph = builder.compile(
    checkpointer=memory
)

print("Advanced LangGraph compiled.")
print("Nodes:", list(graph.get_graph().nodes))


Advanced LangGraph compiled.
Nodes: ['__start__', 'propose', 'approve', 'tools', 'finalize', '__end__']



## 4. Pause for Approval

A graph with interruptions needs a checkpointer
so that its state can be restored when execution
resumes.

Each workflow execution uses a `thread_id`.

We will start a new request and inspect the
paused graph.

At this stage:

- The category proposal exists.
- The graph is waiting for human approval.
- The policy lookup tool has not executed.

We can inspect the pending request and decide
what happens next.

Unlike a normal function call, the graph
does not automatically finish while paused.


In [8]:

thread_a = {
    "configurable": {
        "thread_id": "approval-example-a"
    }
}

initial_result = graph.invoke(
    {
        "request": "My item arrived damaged.",
        "messages": [],
        "audit": []
    },
    config=thread_a
)

snapshot_a = graph.get_state(thread_a)

print("Proposed category:",
      snapshot_a.values["proposed_category"])

print("Next node:", snapshot_a.next)

print("\nPending tool calls:")
print(snapshot_a.values["messages"][-1].tool_calls)

assert snapshot_a.next == ("approve",)
assert snapshot_a.values["proposed_category"] == "damage"

assert not any(
    isinstance(message, ToolMessage)
    for message in snapshot_a.values["messages"]
)

print("\nGraph paused successfully.")
print("The tool has not executed.")


Proposed category: damage
Next node: ('approve',)

Pending tool calls:
[{'name': 'lookup_policy', 'args': {'category': 'damage'}, 'id': 'policy_lookup_1', 'type': 'tool_call'}]

Graph paused successfully.
The tool has not executed.


In [9]:

approved_result = graph.invoke(
    Command(
        resume={
            "approved": True,
            "category": "damage"
        }
    ),
    config=thread_a
)

print("Approval:", approved_result["approved"])
print("Category:", approved_result["proposed_category"])
print("Outcome:", approved_result["outcome"])

print("\nExecution audit:")
for entry in approved_result["audit"]:
    print("-", entry)

assert approved_result["approved"] is True
assert "damaged items" in approved_result["outcome"]

assert any(
    isinstance(message, ToolMessage)
    for message in approved_result["messages"]
)

assert graph.get_state(thread_a).next == ()

print("\nApproved tool execution: PASSED")


Approval: True
Category: damage
Outcome: Customers can request help with damaged items.

Execution audit:
- Proposed policy category: damage
- Human approved category: damage
- Tool result received and finalized

Approved tool execution: PASSED



## 5. Human Rejection and Correction

Human oversight should allow more than approval.

We will demonstrate two additional decisions.

### Rejecting an action

If the reviewer rejects a proposal, the workflow
should finish without executing the tool.

### Correcting an action

Suppose the deterministic classifier chooses
the wrong category.

A reviewer can replace that category before the
tool is executed.

For example:

The customer says "My parcel hasn't arrived."

The basic classifier does not recognize that
wording and proposes the general category.

The reviewer can correct it to "delivery".

This is a realistic reason to allow human edits.


In [10]:

thread_b = {
    "configurable": {
        "thread_id": "rejection-example-b"
    }
}

graph.invoke(
    {
        "request": "I would like a refund.",
        "messages": [],
        "audit": []
    },
    config=thread_b
)

rejected_result = graph.invoke(
    Command(resume={"approved": False}),
    config=thread_b
)

print("Approval:", rejected_result["approved"])
print("Outcome:", rejected_result["outcome"])

assert rejected_result["approved"] is False

assert not any(
    isinstance(message, ToolMessage)
    for message in rejected_result["messages"]
)

print("Rejection correctly prevented tool execution.")


Approval: False
Outcome: Action rejected by human reviewer.
Rejection correctly prevented tool execution.


In [11]:

thread_c = {
    "configurable": {
        "thread_id": "correction-example-c"
    }
}

graph.invoke(
    {
        "request": "My parcel hasn't arrived.",
        "messages": [],
        "audit": []
    },
    config=thread_c
)

before_correction = graph.get_state(thread_c)

print(
    "Original proposal:",
    before_correction.values["proposed_category"]
)

corrected_result = graph.invoke(
    Command(
        resume={
            "approved": True,
            "category": "delivery"
        }
    ),
    config=thread_c
)

print(
    "Corrected category:",
    corrected_result["proposed_category"]
)

print("Tool outcome:", corrected_result["outcome"])

assert before_correction.values["proposed_category"] == "general"
assert corrected_result["proposed_category"] == "delivery"
assert "3–5 business days" in corrected_result["outcome"]

print("\nHuman correction: PASSED")


Original proposal: general
Corrected category: delivery
Tool outcome: Standard delivery takes 3–5 business days.

Human correction: PASSED



## 6. Modify Saved State with update_state()

LangGraph exposes `update_state()` for modifying
checkpointed workflow data.

This can be useful when a human or an external
system needs to correct information before
execution continues.

For example:

1. The graph proposes the wrong category.
2. Execution pauses.
3. A human edits the saved state.
4. The graph continues using the corrected value.

`update_state()` creates a new checkpoint with
the supplied state updates.

It does not mutate historical checkpoints in place.

The optional `as_node` parameter identifies the
node whose output the update should represent.

That affects where the graph continues.

In this demonstration, we update a proposal and
then allow the approval node to run again.


In [12]:

thread_d = {
    "configurable": {
        "thread_id": "manual-update-example-d"
    }
}

graph.invoke(
    {
        "request": "Where is my parcel?",
        "messages": [],
        "audit": []
    },
    config=thread_d
)

before_update = graph.get_state(thread_d)

print(
    "Before update:",
    before_update.values["proposed_category"]
)

# Correct the saved proposal before approving.
graph.update_state(
    thread_d,
    {
        "proposed_category": "delivery",
        "audit": ["Human corrected saved category"]
    },
    as_node="propose"
)

after_update = graph.get_state(thread_d)

print(
    "After update:",
    after_update.values["proposed_category"]
)

print("Next node:", after_update.next)

assert after_update.values["proposed_category"] == "delivery"
assert after_update.next == ("approve",)

# Execute the approval node, which will pause.
graph.invoke(None, config=thread_d)

# Resume using the updated state.
updated_result = graph.invoke(
    Command(resume={"approved": True}),
    config=thread_d
)

print("Final category:", updated_result["proposed_category"])
print("Final outcome:", updated_result["outcome"])

assert updated_result["proposed_category"] == "delivery"
assert "3–5 business days" in updated_result["outcome"]

print("\nManual state update: PASSED")


Before update: general
After update: delivery
Next node: ('approve',)
Final category: delivery
Final outcome: Standard delivery takes 3–5 business days.

Manual state update: PASSED



## 7. Inspect and Replay Historical Checkpoints

LangGraph keeps a history of checkpoints for
each execution thread.

`get_state()` returns the current state.

`get_state_history()` retrieves saved snapshots.

Each snapshot includes information such as:

- State values.
- Checkpoint configuration.
- The next node or nodes scheduled to execute.
- Associated execution metadata.

### What Does Time Travel Mean?

Time travel means selecting an earlier checkpoint
and continuing execution from that point.

It allows us to investigate alternative paths
or reproduce a previous workflow decision.

Important:

- Replaying does not alter the past.
- It creates another continuation from saved state.
- Previously executed external actions are not undone.
- Replaying a checkpoint can execute tools again.

Our example uses a read-only local policy lookup,
so the repeated tool execution is safe.


In [13]:

history = list(
    graph.get_state_history(thread_a)
)

print("Saved checkpoints:", len(history))

for index, snapshot in enumerate(history):
    print(
        f"Checkpoint {index}: "
        f"next={snapshot.next}, "
        f"category={snapshot.values.get('proposed_category')}"
    )

assert len(history) > 1

# Find a checkpoint immediately before tool execution.
tool_checkpoints = [
    snapshot
    for snapshot in history
    if snapshot.next == ("tools",)
]

assert tool_checkpoints, (
    "No pre-tool checkpoint found."
)

replay_checkpoint = tool_checkpoints[0]

print("\nSelected checkpoint:")
print("Next:", replay_checkpoint.next)
print(
    "Checkpoint ID:",
    replay_checkpoint.config["configurable"].get(
        "checkpoint_id"
    )
)

print("Checkpoint selected for replay.")


Saved checkpoints: 6
Checkpoint 0: next=(), category=damage
Checkpoint 1: next=('finalize',), category=damage
Checkpoint 2: next=('tools',), category=damage
Checkpoint 3: next=('approve',), category=damage
Checkpoint 4: next=('propose',), category=None
Checkpoint 5: next=('__start__',), category=None

Selected checkpoint:
Next: ('tools',)
Checkpoint ID: 1f1d3cb1-b531-6a10-8002-cdab12e3c18d
Checkpoint selected for replay.


In [14]:

print("Replaying the approved policy lookup...")

replayed_result = graph.invoke(
    None,
    config=replay_checkpoint.config
)

print("Replayed outcome:", replayed_result["outcome"])

assert replayed_result["approved"] is True
assert "damaged items" in replayed_result["outcome"]

print("\nCheckpoint replay: PASSED")


Replaying the approved policy lookup...
Replayed outcome: Customers can request help with damaged items.

Checkpoint replay: PASSED



## 8. Persistent Checkpoints with SQLite

Until now, we have used `InMemorySaver`.

This stores checkpoints in Python memory.

It is convenient for experiments, but the
checkpoints disappear when the runtime ends.

LangGraph also supports persistent checkpointers.

A persistent checkpointer stores state outside
the Python objects currently in memory.

For this example, we use SQLite.

SQLite is a lightweight database that stores data
in a local file.

We will:

1. Create a SQLite-backed checkpointer.
2. Execute a workflow.
3. Close the database connection.
4. Open a new connection.
5. Recover the completed workflow state.

This demonstrates persistence across database
connections, not across Colab runtime deletion.

The database file itself must be retained if
state is needed in a later environment.


In [15]:

from pathlib import Path
from langgraph.checkpoint.sqlite import SqliteSaver

database_path = Path("langgraph_demo_checkpoints.sqlite")

# Start with a clean demonstration database.
# Do not use this deletion pattern for production data.
if database_path.exists():
    database_path.unlink()

sqlite_thread = {
    "configurable": {
        "thread_id": "sqlite-persistence-example"
    }
}

# First connection: execute and save.
with SqliteSaver.from_conn_string(
    str(database_path)
) as sqlite_memory:

    sqlite_graph = builder.compile(
        checkpointer=sqlite_memory
    )

    sqlite_graph.invoke(
        {
            "request": "I need to return an unused item.",
            "messages": [],
            "audit": []
        },
        config=sqlite_thread
    )

    saved_result = sqlite_graph.invoke(
        Command(
            resume={
                "approved": True,
                "category": "returns"
            }
        ),
        config=sqlite_thread
    )

    assert "30 days" in saved_result["outcome"]

    print("Workflow saved to SQLite.")


# Second connection: recover saved state.
with SqliteSaver.from_conn_string(
    str(database_path)
) as reopened_memory:

    recovered_graph = builder.compile(
        checkpointer=reopened_memory
    )

    recovered = recovered_graph.get_state(
        sqlite_thread
    )

    print("\nRecovered outcome:")
    print(recovered.values["outcome"])

    assert recovered.values["approved"] is True
    assert "30 days" in recovered.values["outcome"]

print("\nSQLite persistence test: PASSED")
print("Database:", database_path.name)


Workflow saved to SQLite.

Recovered outcome:
Unused items can be returned within 30 days.

SQLite persistence test: PASSED
Database: langgraph_demo_checkpoints.sqlite


In [16]:

# Approved workflow
assert approved_result["approved"] is True
assert "damaged items" in approved_result["outcome"]

# Rejected workflow
assert rejected_result["approved"] is False
assert not any(
    isinstance(message, ToolMessage)
    for message in rejected_result["messages"]
)

# Human-corrected workflow
assert corrected_result["proposed_category"] == "delivery"

# Manually updated checkpoint
assert updated_result["proposed_category"] == "delivery"

# Historical checkpoint replay
assert replayed_result["approved"] is True

# Separate execution histories
assert thread_a["configurable"]["thread_id"] != (
    thread_b["configurable"]["thread_id"]
)

# SQLite database exists
assert database_path.is_file()

print("Approved tool execution: PASSED")
print("Rejected action protection: PASSED")
print("Human tool-argument correction: PASSED")
print("Manual checkpoint update: PASSED")
print("Historical checkpoint replay: PASSED")
print("SQLite checkpoint persistence: PASSED")

print("\nFinal verification: PASSED")
print("Model API requests: 0")


Approved tool execution: PASSED
Rejected action protection: PASSED
Human tool-argument correction: PASSED
Manual checkpoint update: PASSED
Historical checkpoint replay: PASSED
SQLite checkpoint persistence: PASSED

Final verification: PASSED
Model API requests: 0



## Findings and Limitations

### What This Notebook Demonstrates

- Building a LangGraph workflow with a real `ToolNode`.
- Creating tool-call messages with structured arguments.
- Pausing execution using `interrupt()`.
- Resuming with `Command(resume=...)`.
- Preventing tool execution after human rejection.
- Correcting tool arguments before execution.
- Defining custom graph state.
- Updating saved checkpoints using `update_state()`.
- Inspecting current and historical graph state.
- Replaying execution from a historical checkpoint.
- Using in-memory and SQLite-backed checkpointing.
- Testing workflow decisions with Python assertions.

### Relationship to Notebook 14

Notebook 14 demonstrated the fundamentals:

- Graph nodes and edges.
- Shared state.
- Conditional routing.
- Validation loops.
- Basic checkpointed memory.

Notebook 15 extends those foundations by introducing
human decisions, state modification and replay.

### Key Lessons

**ToolNode**

A model or application can create a tool request.
LangGraph's ToolNode executes that request and
returns its result to graph state.

**Human oversight**

A graph can pause before executing an action.
Approval and rejection are explicit workflow decisions.

**Manual state updates**

Saved state can be changed without restarting
the entire workflow from its initial input.

**Checkpoint replay**

A previous snapshot can be used to create a new
continuation of a workflow.

**Persistence**

A SQLite checkpointer can recover saved graph state
across database connections.

### Limitations

- The workflow uses deterministic proposal logic,
  not an LLM selecting actions dynamically.
- All policies are fictional, local and read-only.
- Approval decisions are supplied explicitly in
  Python rather than through a separate user interface.
- The example does not perform external transactions.
- In-memory checkpoints are lost when the runtime ends.
- The SQLite database is local to the Colab environment
  unless it is copied to persistent storage.
- Replaying workflows does not undo external actions.
- Repeated execution of side-effecting tools would
  require idempotency and duplicate-action protection.
- No authentication, authorization or production
  approval audit system is implemented.

### Conclusion

This notebook demonstrates advanced LangGraph
execution control beyond ordinary conditional routing.

The workflow pauses for human oversight, allows
approved tool execution, prevents rejected actions,
supports state corrections and retains checkpoints.

It also demonstrates how historical checkpoints
can be inspected and replayed.

These capabilities are useful when developing
stateful agents that must remain inspectable,
recoverable and subject to human control.

The experiments validate LangGraph's workflow
mechanics without requiring external AI services.
